## Cell 1: Setup
**What this demonstrates**: Load API keys and configure the two Anthropic models used — `claude-haiku` for per-chunk context generation (cost-efficient; not a reasoning task) and `claude-sonnet` for final answer generation. Two Chroma collections will be built: one plain, one contextual, for direct comparison.
**Expected output**: ✓ Setup complete with model names, collection names, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

# ── Standard library ─────────────────────────────────────────────────────────
import os
import re
import time
import pathlib
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# ── Constants ─────────────────────────────────────────────────────────────────
# Haiku for context generation: cheaper, fast; situating a chunk is not reasoning
CONTEXT_MODEL  = 'claude-haiku-4-5-20251001'
# Sonnet for answer generation: handles multi-document synthesis
GENERATE_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL    = 'text-embedding-3-small'
CHROMA_PLAIN   = './chroma_contextual_plain'
CHROMA_CTX     = './chroma_contextual_enriched'
CHUNK_SIZE     = 500
CHUNK_OVERLAP  = 50

# ── Client and embeddings ─────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('✓ Setup complete')
print(f'  Context model:  {CONTEXT_MODEL}  (per-chunk, cached)')
print(f'  Generate model: {GENERATE_MODEL}')
print(f'  Embed model:    {EMBED_MODEL}')
print(f'  Collections:    plain ({CHROMA_PLAIN})  |  enriched ({CHROMA_CTX})')
print(f'  Anthropic key:  ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:     ...{_openai_key[-4:]}')

## Cell 2: Data — Two-Document Corpus
**What this demonstrates**: Load two fintech documents whose shared vocabulary creates retrieval ambiguity — both contain phrases like "minimum requirement", "obligation", and "default", but in completely different regulatory contexts. This is the exact failure mode Contextual RAG targets.
**Expected output**: Both document names with sizes, a vocabulary overlap analysis, and the key ambiguous terms that will challenge plain retrieval.

In [ ]:
# ── Locate both documents ─────────────────────────────────────────────────────
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data. Run from repo root or modules/13_contextual_rag/.'

# Two-document corpus — same domain (fintech), different context
DOCS: dict[str, str] = {
    'Meridian Bank Consumer Lending Policy v4.2': (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8'),
    'Basel III Capital Adequacy Framework':        (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8'),
}

print('Two-document corpus loaded:')
for title, text in DOCS.items():
    print(f'  {title!r}')
    print(f'    {len(text):,} chars  |  ~{len(text.split()):,} words')
print()

# ── Vocabulary overlap analysis ───────────────────────────────────────────────
# These terms appear in both documents with completely different meanings —
# exactly the retrieval ambiguity that context-enriched embeddings must resolve.
AMBIGUOUS_TERMS = [
    'minimum requirement',
    'default',
    'capital',
    'obligation',
    'rate',
    'buffer',
]
print('Ambiguous terms appearing in BOTH documents:')
print(f'  {"Term":<22}  {"Lending Policy":>14}  {"Basel III":>10}  Meaning in each')
print('  ' + '-' * 80)
meanings = {
    'minimum requirement': ('Loan eligibility (FICO, DTI)', 'Capital ratios (CET1 4.5%)'),
    'default':             ('Borrower 90 DPD',              'Event triggering capital charge'),
    'capital':             ('Working capital / reserves',   'Regulatory capital (Tier 1/2)'),
    'obligation':          ('Loan repayment duty',          'Payment obligation under Agreement'),
    'rate':                ('APR / FICO tier rate',         'Risk weight / capital ratio'),
    'buffer':              ('Not present',                  'Conservation / countercyclical buffer'),
}
for term in AMBIGUOUS_TERMS:
    texts = list(DOCS.values())
    in_lending = term.lower() in texts[0].lower()
    in_basel   = term.lower() in texts[1].lower()
    m1, m2 = meanings.get(term, ('', ''))
    flag = '✓' if (in_lending and in_basel) else '–'
    print(f'  {flag} {term:<20}  {str(in_lending):>14}  {str(in_basel):>10}  ({m1} | {m2})')
print()
print('Without document context in embeddings, chunks containing these terms score')
print('similarly against each other — retrieval cannot distinguish which document')
print('is relevant to the query.')

## Cell 3: Core — Context Generation and Dual Indexing
**What this demonstrates**: For each chunk, extract the section name heuristically, then call Claude Haiku (with prompt caching on the full document) to generate a 50–100 token situating context. Build two Chroma collections — plain and enriched — so Cell 5 can run a direct comparison.
**Expected output**: Per-chunk progress with context previews, total chunks indexed in both collections, cost-saving estimate from prompt caching.

In [ ]:
# ── Splitter ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    separators=['\n================================================================================\n', '\n\n', '\n', '. ', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


# ── Section name extractor ────────────────────────────────────────────────────
def extract_section_name(chunk_text: str, full_doc: str) -> str:
    """Find the most recent section/article header preceding this chunk.

    Searches backwards from the chunk's position in the source document for
    SECTION, Article, or PART headers. Returns the last match before the chunk,
    which is the structural context the chunk belongs to.

    Args:
        chunk_text: Raw text of the chunk.
        full_doc:   Full source document text.

    Returns:
        Section/article name string, or 'Document preamble' if none found.
    """
    pos = full_doc.find(chunk_text.strip()[:60])
    if pos == -1:
        return 'Unknown section'
    preceding = full_doc[:pos]
    for pattern in [
        r'(?:SECTION \d+[:\s][A-Z][^\n]+)',
        r'(?:Article \d+[^\n]+)',
        r'(?:PART [IVX]+[:\s][^\n]+)',
    ]:
        matches = re.findall(pattern, preceding)
        if matches:
            return matches[-1].strip()[:80]
    return 'Document preamble'


# ── Context generator (with prompt caching) ───────────────────────────────────
def generate_chunk_context(
    chunk_text:   str,
    doc_title:    str,
    doc_text:     str,
    section_name: str,
) -> str:
    """Call Claude Haiku to generate a situating context for this chunk.

    The full document is marked cache_control=ephemeral so it is cached across
    all N per-chunk calls for this document. After the first call, only the
    chunk-specific portion incurs input token cost (~85–90% savings per call).

    Args:
        chunk_text:   Raw chunk to situate.
        doc_title:    Human-readable document name.
        doc_text:     Full source document (cached across calls).
        section_name: Section header extracted by extract_section_name().

    Returns:
        50–100 token context string describing the chunk's position in the document.

    Fintech note:
        For production corpora, parallelise calls with asyncio.gather() and
        respect the Anthropic rate limit (default 5 req/s for Haiku). The cached
        document block counts toward the cache write cost only on the first call.
    """
    response = client.messages.create(
        model=CONTEXT_MODEL,
        max_tokens=150,
        system=[
            {
                'type': 'text',
                'text': (
                    'You are a document analyst. Given a document and a specific chunk, '
                    'write 2–3 sentences that situate the chunk within the document. '
                    'Include: (1) the document name and type, (2) which section or article '
                    'this chunk belongs to, (3) the concept or clause it elaborates. '
                    'Describe position, not content. Be specific and concise.'
                ),
                'cache_control': {'type': 'ephemeral'},   # cache the system prompt
            }
        ],
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        # Full document cached: identical across all chunks in this document
                        'text': f'<document title="{doc_title}">\n{doc_text}\n</document>',
                        'cache_control': {'type': 'ephemeral'},
                    },
                    {
                        'type': 'text',
                        'text': (
                            f'Situate this chunk from "{doc_title}", {section_name}:\n\n'
                            f'<chunk>\n{chunk_text}\n</chunk>'
                        ),
                    },
                ],
            }
        ],
    )
    return response.content[0].text.strip()


# ── Enriched chunk builder ────────────────────────────────────────────────────
def build_enriched_chunk(
    doc_title:    str,
    section_name: str,
    context:      str,
    chunk_text:   str,
) -> str:
    """Prepend the generated context to the raw chunk text before embedding.

    The resulting string is what gets embedded — not the raw chunk alone.
    The vector now encodes document-level position alongside chunk content.
    The raw chunk is preserved separately in metadata for the generation prompt.
    """
    return (
        f'Document: {doc_title}\n'
        f'Section: {section_name}\n'
        f'Context: {context}\n\n'
        f'Chunk content: {chunk_text}'
    )


# ── Build both collections ────────────────────────────────────────────────────
plain_docs:      list[Document] = []
contextual_docs: list[Document] = []

total_chunks = 0
t_start = time.perf_counter()

for doc_title, doc_text in DOCS.items():
    raw_chunks = splitter.split_text(doc_text)
    print(f'\n── {doc_title} ({len(raw_chunks)} chunks) ──────────────────────────')

    for i, chunk_text in enumerate(raw_chunks):
        section_name = extract_section_name(chunk_text, doc_text)

        # Plain document: raw chunk only
        plain_docs.append(Document(
            page_content=chunk_text,
            metadata={'doc_title': doc_title, 'section': section_name, 'chunk_idx': i},
        ))

        # Contextual document: Claude-generated context prepended before embedding
        context = generate_chunk_context(chunk_text, doc_title, doc_text, section_name)
        enriched_text = build_enriched_chunk(doc_title, section_name, context, chunk_text)

        contextual_docs.append(Document(
            page_content=enriched_text,
            # Store raw chunk in metadata — use this for the generation prompt, not enriched text
            metadata={'doc_title': doc_title, 'section': section_name, 'chunk_idx': i, 'raw_chunk': chunk_text},
        ))

        ctx_preview = context[:70].replace('\n', ' ')
        print(f'  [{i+1:02d}/{len(raw_chunks):02d}]  {section_name[:35]:<35}  ctx: {ctx_preview!r}...')
        total_chunks += 1

index_ms = (time.perf_counter() - t_start) * 1000

# ── Embed and store both collections ─────────────────────────────────────────
plain_store = Chroma.from_documents(
    documents=plain_docs,
    embedding=embeddings,
    collection_name='contextual_plain',
    persist_directory=CHROMA_PLAIN,
)
ctx_store = Chroma.from_documents(
    documents=contextual_docs,
    embedding=embeddings,
    collection_name='contextual_enriched',
    persist_directory=CHROMA_CTX,
)

print(f'\nIndexed: {total_chunks} total chunks in {index_ms/1000:.1f}s')
print(f'  Plain collection:      {plain_store._collection.count()} chunks')
print(f'  Contextual collection: {ctx_store._collection.count()} chunks')
print()
print('Prompt caching benefit:')
doc_tokens_approx = sum(len(t.split()) * 1.3 for t in DOCS.values())
print(f'  Without caching: ~{int(doc_tokens_approx * total_chunks):,} input tokens (full doc re-sent per chunk)')
print(f'  With caching:    ~{int(doc_tokens_approx * len(DOCS)):,} input tokens (doc cached; only chunks re-sent)')
print(f'  Estimated savings: ~{100 * (1 - len(DOCS) / total_chunks):.0f}%')

## Cell 4: Run — Ambiguous Multi-Document Query
**What this demonstrates**: Query both collections with an intentionally ambiguous question — "What is the minimum capital requirement?" — which matches vocabulary in both documents but refers to different concepts. Contextual retrieval attributes the answer to the correct document; plain retrieval may return a mix.
**Expected output**: Top results from each collection with document attribution, generated answer with source citations, and latency.

In [ ]:
QUERY = 'What is the minimum capital requirement?'

# ── Query both collections ────────────────────────────────────────────────────
t0 = time.perf_counter()
plain_hits = plain_store.similarity_search_with_score(QUERY, k=5)
ctx_hits   = ctx_store.similarity_search_with_score(QUERY, k=5)
retrieval_ms = (time.perf_counter() - t0) * 1000

print(f'Query: {QUERY!r}')
print()

# ── Plain retrieval results ────────────────────────────────────────────────────
print('PLAIN retrieval (raw chunks, no context prefix):')
for i, (doc, dist) in enumerate(plain_hits, 1):
    sim = 1.0 - dist / 2.0
    src = doc.metadata.get('doc_title', '?')[:40]
    sec = doc.metadata.get('section', '?')[:35]
    preview = doc.page_content[:80].replace('\n', ' ')
    print(f'  [{i}] sim={sim:.3f}  [{src}]  {sec}')
    print(f'       {preview!r}...')
print()

# ── Contextual retrieval results ──────────────────────────────────────────────
print('CONTEXTUAL retrieval (context-enriched embeddings):')
for i, (doc, dist) in enumerate(ctx_hits, 1):
    sim = 1.0 - dist / 2.0
    src = doc.metadata.get('doc_title', '?')[:40]
    sec = doc.metadata.get('section', '?')[:35]
    # Show the raw chunk from metadata (not the enriched embedding text)
    raw = doc.metadata.get('raw_chunk', doc.page_content)[:80].replace('\n', ' ')
    print(f'  [{i}] sim={sim:.3f}  [{src}]  {sec}')
    print(f'       {raw!r}...')
print()

# ── Generate answer from contextual top-3 ────────────────────────────────────
# Use raw_chunk from metadata — the LLM should not receive the context header twice
context = '\n\n---\n\n'.join(
    f'[Source {i+1}: {doc.metadata.get("doc_title", "?")}, {doc.metadata.get("section", "?")}]\n'
    f'{doc.metadata.get("raw_chunk", doc.page_content)}'
    for i, (doc, _) in enumerate(ctx_hits[:3])
)

SYSTEM = (
    'You are a regulatory compliance analyst. '
    'Answer using ONLY the provided excerpts. '
    'For each claim, cite the document name and section in brackets. '
    'If the query is ambiguous across documents, answer for each separately.'
)

t1 = time.perf_counter()
response = client.messages.create(
    model=GENERATE_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{'role': 'user', 'content': f'Excerpts:\n{context}\n\nQuestion: {QUERY}'}],
)
answer: str = response.content[0].text
gen_ms = (time.perf_counter() - t1) * 1000

print('=' * 65)
print('ANSWER (from contextual retrieval)')
print('=' * 65)
print(answer)
print('=' * 65)
print()
print(f'Retrieval: {retrieval_ms:.0f} ms  |  Generation: {gen_ms:.0f} ms  |  Total: {retrieval_ms + gen_ms:.0f} ms')

## Cell 5: Inspect — Context Prefixes and Disambiguation
**What this demonstrates**: Expose the full context prefix that was prepended to each chunk before embedding. Show side-by-side how plain vs contextual embeddings rank the same chunks differently, and why the context header is the mechanism that resolves document-level ambiguity.
**Expected output**: Full context prefix for each top contextual result, a plain-vs-contextual ranking comparison, and a document attribution summary showing how many results came from each source.

In [ ]:
# ── Show full context prefixes ────────────────────────────────────────────────
# This is what was embedded — the context header + raw chunk.
# The header is invisible at query time but its meaning is encoded in the vector.
print(f'Query: {QUERY!r}')
print()
print('Context prefixes embedded with each top-3 contextual chunk:')
print('(These strings were embedded — not the raw chunk alone)')
print()
for i, (doc, dist) in enumerate(ctx_hits[:3], 1):
    sim = 1.0 - dist / 2.0
    # The enriched text is the full page_content of the contextual document
    enriched = doc.page_content
    raw      = doc.metadata.get('raw_chunk', '')
    # The context prefix is everything before 'Chunk content:'
    if 'Chunk content:' in enriched:
        prefix = enriched[:enriched.index('Chunk content:')].strip()
    else:
        prefix = enriched[:200]
    print(f'  [{i}] sim={sim:.3f}  ({len(raw)} char raw chunk  →  {len(enriched)} char enriched)')
    print(f'  Prefix embedded:')
    for line in prefix.splitlines():
        print(f'    {line}')
    print(f'  Raw chunk (first 80 chars): {raw[:80].replace(chr(10), " ")!r}...')
    print()

# ── Plain vs contextual ranking comparison ────────────────────────────────────
print('── RANKING COMPARISON ───────────────────────────────────────────────────────')
print(f'  {"Rank":<4}  {"Plain retrieval":<38}  Contextual retrieval')
print('  ' + '-' * 90)
for rank in range(5):
    plain_col = ctx_col = ''
    if rank < len(plain_hits):
        doc, dist = plain_hits[rank]
        sim = 1.0 - dist / 2.0
        src = doc.metadata.get('doc_title', '?')[:18]
        plain_col = f'sim={sim:.3f}  [{src}]'
    if rank < len(ctx_hits):
        doc, dist = ctx_hits[rank]
        sim = 1.0 - dist / 2.0
        src = doc.metadata.get('doc_title', '?')[:18]
        ctx_col = f'sim={sim:.3f}  [{src}]'
    print(f'  {rank+1:<4}  {plain_col:<38}  {ctx_col}')

# ── Document attribution summary ──────────────────────────────────────────────
print()
print('Document attribution (top-5 results):')
for label, hits in [('Plain', plain_hits), ('Contextual', ctx_hits)]:
    counts: dict[str, int] = {}
    for doc, _ in hits:
        src = doc.metadata.get('doc_title', 'Unknown')[:35]
        counts[src] = counts.get(src, 0) + 1
    print(f'  {label}:')
    for src, count in sorted(counts.items(), key=lambda x: -x[1]):
        bar = '█' * count
        print(f'    {count}× {bar}  {src!r}')

print()
print('Contextual RAG effect:')
print('  Context-enriched embeddings encode document identity into the vector.')
print('  "Minimum capital requirement" in a Basel III context scores higher against')
print('  a query about capital requirements than the same phrase in a lending policy.')

## Cell 6: Fintech — Cross-Policy Compliance Query
**What this demonstrates**: A compliance analyst's query that legitimately spans both documents — "What happens when a borrower fails to meet payment obligations?" The answer has two distinct components: the bank's internal remediation process (Lending Policy) and the regulatory capital treatment of the defaulted loan (Basel III). Contextual RAG attributes each retrieved chunk to its source, enabling a complete cross-policy answer.
**Expected output**: Retrieved chunks attributed to both documents, a synthesised compliance answer citing both policies, and a comparison showing which chunks plain retrieval missed.

In [ ]:
# ── Cross-policy compliance query ─────────────────────────────────────────────
# A compliance analyst needs BOTH answers:
#   1. What does the lending policy say about borrower default? (Section 8)
#   2. How does Basel III require the bank to treat the defaulted loan? (Article 8, 150% RW)
# Plain retrieval likely returns only the stronger lexical match.
# Contextual retrieval attributes chunks correctly, enabling cross-policy synthesis.
COMPLIANCE_QUERY = (
    'What happens when a borrower fails to meet payment obligations? '
    'What are the regulatory capital implications?'
)

COMPLIANCE_SYSTEM = (
    'You are a bank compliance officer preparing a regulatory briefing. '
    'Answer using ONLY the provided excerpts. '
    'Structure your answer in two parts: '
    '(1) Internal bank procedures (from the Lending Policy), '
    '(2) Regulatory capital treatment (from Basel III). '
    'Cite the document name and section for every claim. '
    'If either part is not covered by the provided excerpts, say so explicitly.'
)

# ── Query both collections ─────────────────────────────────────────────────────
plain_compliance   = plain_store.similarity_search_with_score(COMPLIANCE_QUERY, k=5)
ctx_compliance     = ctx_store.similarity_search_with_score(COMPLIANCE_QUERY, k=5)

print(f'Query: {COMPLIANCE_QUERY!r}')
print()

# ── Show which documents each retriever found ──────────────────────────────────
for label, hits in [('Plain', plain_compliance), ('Contextual', ctx_compliance)]:
    doc_sources = [h[0].metadata.get('doc_title', '?')[:35] for h in hits]
    unique_docs = sorted(set(doc_sources))
    print(f'{label} retrieval — documents represented: {len(unique_docs)}')
    for doc_name in unique_docs:
        count = doc_sources.count(doc_name)
        print(f'  {count}× {doc_name!r}')
    print()

# ── Generate cross-policy answer from contextual top-5 ───────────────────────
context_compliance = '\n\n---\n\n'.join(
    f'[Source {i+1}: {doc.metadata.get("doc_title", "?")}, {doc.metadata.get("section", "?")}]\n'
    f'{doc.metadata.get("raw_chunk", doc.page_content)}'
    for i, (doc, _) in enumerate(ctx_compliance)
)

t_gen = time.perf_counter()
response_c = client.messages.create(
    model=GENERATE_MODEL,
    max_tokens=700,
    system=COMPLIANCE_SYSTEM,
    messages=[{'role': 'user', 'content': f'Policy excerpts:\n{context_compliance}\n\nQuestion: {COMPLIANCE_QUERY}'}],
)
gen_ms = (time.perf_counter() - t_gen) * 1000
answer_c: str = response_c.content[0].text

print('=' * 65)
print('CROSS-POLICY COMPLIANCE BRIEFING')
print('=' * 65)
print(answer_c)
print('=' * 65)

# ── Gap analysis: what plain retrieval misses ──────────────────────────────────
print()
print('Cross-policy coverage gap (plain vs contextual):')
plain_doc_set = {d.metadata.get('doc_title', '')[:35] for d, _ in plain_compliance}
ctx_doc_set   = {d.metadata.get('doc_title', '')[:35] for d, _ in ctx_compliance}
only_in_ctx   = ctx_doc_set - plain_doc_set
if only_in_ctx:
    print(f'  Documents found by contextual only: {only_in_ctx}')
    print('  → Plain retrieval gave an incomplete cross-policy answer.')
else:
    print('  Both retrievers found the same documents for this query.')
    print('  Contextual advantage shows in ranking precision — check Cell 5 scores.')
print()
print('Production recommendation:')
print('  Contextual RAG + BM25 (Hybrid) is Anthropic\'s recommended combination.')
print('  Contextual embeddings improve dense recall; BM25 catches exact term matches.')
print('  Together: 49% fewer retrieval failures vs plain dense (Anthropic, 2024).')
print()
print(f'Generation time: {gen_ms:.0f} ms')